# 🎙️ Sir C.V. Raman Marathon — Chatterbox Voice Cloner (Test)
Proven 1-click script from `kaggle_voice_cloner.ipynb`!

### Setup:
1. Accelerator: **GPU T4 x2** (Internet: ON)
2. Click **Run All**

In [ ]:
# STEP 1: Install Dependencies & Verify GPU
!pip install -q chatterbox-tts torchvision==0.21.0 yt-dlp "setuptools<81"
import torch, torchvision
torch.backends.cudnn.enabled = False
assert torch.cuda.is_available(), "❌ GPU is OFF! Enable Accelerator -> GPU in Kaggle Settings."
print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)} | torch={torch.__version__} | tv={torchvision.__version__}")


In [ ]:
# STEP 2: Prepare Clean 20s Voice Reference
import os, glob, subprocess

YOUTUBE_URL = "https://youtu.be/ixCAZT2vHcM"
os.makedirs("/kaggle/working/ref", exist_ok=True)
clean_ref = "/kaggle/working/clean_voice_ref.wav"

uploaded = glob.glob("/kaggle/input/**/intro_0_20.wav", recursive=True) + glob.glob("/kaggle/working/intro_0_20.wav")
if uploaded:
    raw_wav = uploaded[0]
    print(f"✅ Found uploaded sample: {raw_wav}")
else:
    print("⏳ Downloading clean 0:00-0:20 voice sample from YouTube...")
    raw_wav = "/kaggle/working/ref/intro_0_20.wav"
    subprocess.run([
        "yt-dlp", "-f", "ba", "--download-sections", "*0:00-0:20",
        "-x", "--audio-format", "wav",
        "-o", "/kaggle/working/ref/intro_0_20.%(ext)s",
        YOUTUBE_URL
    ], check=True)

subprocess.run(["ffmpeg", "-y", "-i", raw_wav, "-ac", "1", "-ar", "24000", "-af", "highpass=f=80,loudnorm=I=-16:TP=-1.5:LRA=11", clean_ref], check=True)
print(f"🎉 Clean Reference Ready: {clean_ref}")


In [ ]:
# STEP 3: Load Chatterbox Multilingual Voice Cloning Model
import torch
torch.backends.cudnn.enabled = False
import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
print("✅ Chatterbox Multilingual Model Loaded!")


In [ ]:
# STEP 4: Quick 1-Clip Verification (Only Q2 Answer - 10 Seconds!)
import os, torchaudio as ta
from IPython.display import Audio, display

# Pure Devanagari ending with single purna viram (No English, No '!')
clean_text = "बिल्कुल सही। प्रकाश संश्लेषण में पौधे कार्बन डाइऑक्साइड ग्रहण करते हैं और ऑक्सीजन गैस छोड़ते हैं। सही उत्तर है विकल्प डी, कार्बन डाइऑक्साइड।"

print("🎙️ Generating 1 Quick Test Clip...")
wav = model.generate(
    text=clean_text,
    language_id="hi",
    audio_prompt_path="/kaggle/working/clean_voice_ref.wav",
    exaggeration=0.45,
    cfg_weight=0.55,
    temperature=0.45,
    repetition_penalty=2.0
)

out_path = "/kaggle/working/test_q002_answer.wav"
ta.save(out_path, wav, model.sr)
print("✅ Done in ~8 seconds! Listen directly below:")
display(Audio(out_path))
